In [1]:
import datetime as dt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import interp1d
import statsmodels.api as sm
import yfinance as yf

In [3]:
N_SIMS = 200_000  # Número de simulaciones de Monte Carlo
N_YEARS = 10  # Horizonte del DCF en años
DT_SIM = 1 / 12  # Paso de tiempo mensual
WACC = 0.10  # Tasa de descuento r = 10%

OIL_PROD_ANNUAL = 1_000_000  # Barriles/año (bbl/año)
GAS_PROD_ANNUAL = 5_000_000  # Millones de BTU/año (mmBTU/año)

In [4]:
HIST_START = '2005-01-01'
HIST_END = '2026-11-01'

In [5]:
raw_data = yf.download(
    ['CL=F', 'NG=F'],
    start=HIST_START,
    end=HIST_END,
    interval='1mo',
    progress=False,
)['Close']

In [6]:
df_prices = raw_data.rename(columns={'CL=F': 'Oil', 'NG=F': 'Gas'}).dropna()
df_prices

Ticker,Oil,Gas
Date,,
2005-01-01,48.200001,6.321
2005-02-01,51.750000,6.730
2005-03-01,55.400002,7.653
2005-04-01,49.720001,6.585
2005-06-01,56.500000,6.981
...,...,...
2026-06-01,69.500000,3.275
2026-07-01,84.669998,2.747
2026-08-01,85.760002,2.935


In [8]:
def calibrate_ou_dynamics(price_series, dt_hist=1 / 12):
    """Estima theta (velocidad de reversión) y sigma (volatilidad) del proceso OU

    logarítmico mediante regresión AR(1) exacta: X_t = a + b * X_{t-1} + eps_t.
    """
    X = np.log(price_series.values)
    X_t = X[1:]
    X_prev = sm.add_constant(X[:-1])
    model = sm.OLS(X_t, X_prev).fit()

    a, b = model.params
    resid = model.resid
    sd_eps = np.std(resid, ddof=2)

    if not (0 < b < 1):
        print(
            f'Advertencia: b={b:.4f} fuera de (0,1). La serie muestra raíz unitaria o tendencia.'
        )

    theta = -np.log(b) / dt_hist
    sigma = sd_eps * np.sqrt((-2 * np.log(b)) / (dt_hist * (1 - b**2)))

    return {
        'theta': theta,
        'sigma': sigma,
        'spot_hist': float(price_series.iloc[-1]),
        'resid': resid,
    }

In [9]:
oil_params = calibrate_ou_dynamics(df_prices['Oil'], dt_hist=1 / 12)
gas_params = calibrate_ou_dynamics(df_prices['Gas'], dt_hist=1 / 12)
rho = float(np.corrcoef(oil_params['resid'], gas_params['resid'])[0, 1])

print('=== 1. PARÁMETROS DINÁMICOS CALIBRADOS (HISTÓRICO) ===')
print(
    f"Petróleo (WTI) -> Spot ref: ${oil_params['spot_hist']:.2f} | theta: {oil_params['theta']:.4f} | sigma: {oil_params['sigma']:.2%}"
)
print(
    f"Gas Natural    -> Spot ref: ${gas_params['spot_hist']:.2f} | theta: {gas_params['theta']:.4f} | sigma: {gas_params['sigma']:.2%}"
)
print(f'Correlación entre choques mensuales (rho): {rho:.4f}\n')

=== 1. PARÁMETROS DINÁMICOS CALIBRADOS (HISTÓRICO) ===
Petróleo (WTI) -> Spot ref: $104.12 | theta: 1.1372 | sigma: 45.04%
Gas Natural    -> Spot ref: $2.90 | theta: 0.8449 | sigma: 58.26%
Correlación entre choques mensuales (rho): 0.0114



In [21]:
def build_monthly_futures_curve(
    asset_name,
    root_ticker,
    spot_fallback,
    n_years,
    dt_sim=1 / 12,
):
    """Descarga contratos futuros activos en NYMEX (Yahoo Finance) e interpola

    el vector mensual F(0, t) para t = 0, 1, ..., N_YEARS*12 meses.
    """
    total_steps = int(round(n_years / dt_sim))
    time_grid = np.linspace(0, n_years, total_steps + 1)

    month_codes = {
        1: 'F',
        2: 'G',
        3: 'H',
        4: 'J',
        5: 'K',
        6: 'M',
        7: 'N',
        8: 'Q',
        9: 'U',
        10: 'V',
        11: 'X',
        12: 'Z',
    }
    today = dt.date.today()
    max_months_fetch = min(total_steps, 72)  # Hasta 6 años de contratos

    contract_tickers = []
    maturities_years = []

    for m_ahead in range(1, max_months_fetch + 1):
        target_year = today.year + (today.month + m_ahead - 1) // 12
        target_month = (today.month + m_ahead - 1) % 12 + 1
        yy = str(target_year)[-2:]
        ticker = f'{root_ticker}{month_codes[target_month]}{yy}.NYM'
        contract_tickers.append(ticker)
        maturities_years.append(m_ahead * dt_sim)

    data_fut = yf.download(
        contract_tickers, period='5d', interval='1d', progress=False
    )['Close']

    valid_t = []
    valid_f = []

    for ticker, t_mat in zip(contract_tickers, maturities_years):
        if ticker in data_fut.columns:
            s = data_fut[ticker].dropna()
            if not s.empty and float(s.iloc[-1]) > 0:
                valid_t.append(t_mat)
                valid_f.append(float(s.iloc[-1]))

    # Anclar t=0 al primer futuro cercano (o spot histórico si no hubo descarga)
    if valid_f:
        valid_t = [0.0] + valid_t
        valid_f = [valid_f[0]] + valid_f
    else:
        valid_t = [0.0, n_years]
        valid_f = [spot_fallback, spot_fallback]

    source_label = f'NYMEX/YF ({len(valid_t)-1} contratos activos)'

    # Interpolación lineal entre contratos y extrapolación plana más allá del último vencimiento
    interp_func = interp1d(
        valid_t,
        valid_f,
        kind='linear',
        bounds_error=False,
        fill_value=(valid_f[0], valid_f[-1]),
    )
    F_curve = interp_func(time_grid)

    print(
        f'Curva {asset_name} [{source_label}]: F(0) = ${F_curve[0]:.2f} | F(1a) = ${F_curve[12]:.2f} | F(5a) = ${F_curve[60]:.2f} | F(10a) = ${F_curve[-1]:.2f}'
    )
    return time_grid, F_curve

In [22]:
print('=== 2. CURVAS DE FUTUROS CONSTRUIDAS ===')
time_grid, F_oil_curve = build_monthly_futures_curve(
    'Oil',
    'CL',
    oil_params['spot_hist'],
    N_YEARS,
    DT_SIM,
)
_, F_gas_curve = build_monthly_futures_curve(
    'Gas',
    'NG',
    gas_params['spot_hist'],
    N_YEARS,
    DT_SIM,
)
print()

=== 2. CURVAS DE FUTUROS CONSTRUIDAS ===


$CLG32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$CLH32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$CLK30.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$CLU29.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$CLV29.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$CLK32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$CLX32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$CLM32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$CLQ31.NYM: possibly delisted; no price 

Curva Oil [NYMEX/YF (47 contratos activos)]: F(0) = $83.98 | F(1a) = $71.02 | F(5a) = $61.24 | F(10a) = $59.39


$NGZ31.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$NGF32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$NGU32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$NGN32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$NGQ32.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")

5 Failed downloads:
['NGZ31.NYM', 'NGF32.NYM', 'NGU32.NYM', 'NGN32.NYM', 'NGQ32.NYM']: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


Curva Gas [NYMEX/YF (67 contratos activos)]: F(0) = $3.17 | F(1a) = $3.29 | F(5a) = $3.70 | F(10a) = $3.59



In [23]:
def compute_monthly_equilibrium_mu(F_curve, theta, sigma, dt_sim):
    """Despeja el vector mu_k (k = 1, ..., total_steps) para el proceso OU escalonado

    tal que E[exp(X_k)] = F_curve[k] en cada mes.
    """
    total_steps = len(F_curve) - 1
    mu_vec = np.zeros(total_steps)

    exp_term = np.exp(-theta * dt_sim)
    one_minus_exp = 1.0 - exp_term

    # Estado inicial determinista X_0 = ln(F(0, 0)) -> Var(X_0) = 0
    E_X_prev = np.log(F_curve[0])

    for k in range(1, total_steps + 1):
        t_k = k * dt_sim
        # Varianza acumulada exacta del proceso OU en el instante t_k
        var_X_k = (sigma**2 / (2.0 * theta)) * (1.0 - np.exp(-2.0 * theta * t_k))

        # Esperanza objetivo de X_k ajustada por convexidad lognormal
        E_X_target = np.log(F_curve[k]) - 0.5 * var_X_k

        # Despeje algebraico exacto de mu_k en la solución discreta mensual
        mu_vec[k - 1] = (E_X_target - E_X_prev * exp_term) / one_minus_exp
        E_X_prev = E_X_target

    return mu_vec


mu_oil_vec = compute_monthly_equilibrium_mu(
    F_oil_curve, oil_params['theta'], oil_params['sigma'], DT_SIM
)
mu_gas_vec = compute_monthly_equilibrium_mu(
    F_gas_curve, gas_params['theta'], gas_params['sigma'], DT_SIM
)

In [24]:
steps_per_year = int(round(1 / DT_SIM))
total_steps = N_YEARS * steps_per_year

np.random.seed(42)

X_oil = np.zeros((total_steps + 1, N_SIMS))
X_gas = np.zeros((total_steps + 1, N_SIMS))

X_oil[0] = np.log(F_oil_curve[0])
X_gas[0] = np.log(F_gas_curve[0])

exp_oil = np.exp(-oil_params['theta'] * DT_SIM)
std_oil = oil_params['sigma'] * np.sqrt(
    (1.0 - np.exp(-2.0 * oil_params['theta'] * DT_SIM))
    / (2.0 * oil_params['theta'])
)

exp_gas = np.exp(-gas_params['theta'] * DT_SIM)
std_gas = gas_params['sigma'] * np.sqrt(
    (1.0 - np.exp(-2.0 * gas_params['theta'] * DT_SIM))
    / (2.0 * gas_params['theta'])
)

for k in range(1, total_steps + 1):
    Z1 = np.random.standard_normal(N_SIMS)
    Z2 = np.random.standard_normal(N_SIMS)

    Z_oil = Z1
    Z_gas = rho * Z1 + np.sqrt(1.0 - rho**2) * Z2

    # Solución exacta mensual usando el equilibrio mu_k correspondiente al mes k
    X_oil[k] = (
        X_oil[k - 1] * exp_oil
        + mu_oil_vec[k - 1] * (1.0 - exp_oil)
        + std_oil * Z_oil
    )
    X_gas[k] = (
        X_gas[k - 1] * exp_gas
        + mu_gas_vec[k - 1] * (1.0 - exp_gas)
        + std_gas * Z_gas
    )

# Transformación a precios en dólares P_t = exp(X_t)
P_oil_paths = np.exp(X_oil)
P_gas_paths = np.exp(X_gas)

# Promedio anual de precios realizados para los flujos de caja del DCF
P_oil_annual = P_oil_paths[1:].reshape(N_YEARS, steps_per_year, N_SIMS).mean(axis=1)
P_gas_annual = P_gas_paths[1:].reshape(N_YEARS, steps_per_year, N_SIMS).mean(axis=1)

In [25]:
FCF_annual = (P_oil_annual * OIL_PROD_ANNUAL) + (P_gas_annual * GAS_PROD_ANNUAL)

years_vec = np.arange(1, N_YEARS + 1).reshape(-1, 1)
discount_factors = 1.0 / ((1.0 + WACC) ** years_vec)

# NPV por trayectoria en Millones de USD
NPV_distribution = (FCF_annual * discount_factors).sum(axis=0) / 1e6

mean_npv = np.mean(NPV_distribution)
median_npv = np.median(NPV_distribution)
p10_npv = np.percentile(NPV_distribution, 10)
p90_npv = np.percentile(NPV_distribution, 90)
std_npv = np.std(NPV_distribution)

print(
    f'=== 3. RESULTADOS DEL DCF ({N_SIMS:,} SIMULACIONES A {N_YEARS} AÑOS, WACC={WACC:.0%}) ==='
)
print(f'NPV Esperado (Media):   ${mean_npv:,.2f} M')
print(f'NPV Mediano (P50):      ${median_npv:,.2f} M')
print(f'Escenario Bajo (P10):   ${p10_npv:,.2f} M')
print(f'Escenario Alto (P90):   ${p90_npv:,.2f} M')
print(f'Desviación Estándar:    ${std_npv:,.2f} M')

=== 3. RESULTADOS DEL DCF (200,000 SIMULACIONES A 10 AÑOS, WACC=10%) ===
NPV Esperado (Media):   $506.00 M
NPV Mediano (P50):      $502.91 M
Escenario Bajo (P10):   $441.28 M
Escenario Alto (P90):   $574.79 M
Desviación Estándar:    $52.43 M


In [28]:
n_paths_plot = 70

# --- GRÁFICA 1: Trayectorias vs Curva de Futuros (Petróleo y Gas) ---
fig_paths = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        'Petróleo WTI: Simulación Monte Carlo vs. Curva de Futuros NYMEX',
        'Gas Natural Henry Hub: Simulación Monte Carlo vs. Curva de Futuros NYMEX',
    ),
)

# Petróleo
for i in range(n_paths_plot):
    fig_paths.add_trace(
        go.Scatter(
            x=time_grid,
            y=P_oil_paths[:, i],
            mode='lines',
            line=dict(color='rgba(0, 191, 255, 0.14)', width=1),
            showlegend=(i == 0),
            name='Trayectorias MC (muestra)',
            hoverinfo='skip',
        ),
        row=1,
        col=1,
    )

fig_paths.add_trace(
    go.Scatter(
        x=time_grid,
        y=P_oil_paths.mean(axis=1),
        mode='lines',
        line=dict(color='#ffffff', width=3),
        name='Media Monte Carlo E[P_t]',
    ),
    row=1,
    col=1,
)

fig_paths.add_trace(
    go.Scatter(
        x=time_grid,
        y=F_oil_curve,
        mode='lines',
        line=dict(color='#ff5252', width=2.5, dash='dash'),
        name='Curva de Futuros F(0,t)',
    ),
    row=1,
    col=1,
)

# Gas Natural
for i in range(n_paths_plot):
    fig_paths.add_trace(
        go.Scatter(
            x=time_grid,
            y=P_gas_paths[:, i],
            mode='lines',
            line=dict(color='rgba(0, 230, 118, 0.14)', width=1),
            showlegend=False,
            hoverinfo='skip',
        ),
        row=2,
        col=1,
    )

fig_paths.add_trace(
    go.Scatter(
        x=time_grid,
        y=P_gas_paths.mean(axis=1),
        mode='lines',
        line=dict(color='#ffffff', width=3),
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig_paths.add_trace(
    go.Scatter(
        x=time_grid,
        y=F_gas_curve,
        mode='lines',
        line=dict(color='#ff5252', width=2.5, dash='dash'),
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig_paths.update_yaxes(title_text='Precio ($/bbl)', row=1, col=1)
fig_paths.update_yaxes(title_text='Precio ($/mmBTU)', row=2, col=1)
fig_paths.update_xaxes(title_text='Horizonte (Años)', row=2, col=1)

fig_paths.update_layout(
    title='Simulación OU No-Homogénea Calibrada a Curva de Futuros',
    template='plotly_dark',
    width=950,
    height=700,
    hovermode='x unified',
)
fig_paths.show()

# --- GRÁFICA 2: Distribución del NPV ---
fig_npv = go.Figure()

fig_npv.add_trace(
    go.Histogram(
        x=NPV_distribution,
        nbinsx=75,
        marker_color='#3b82f6',
        marker_line_color='#111111',
        marker_line_width=0.8,
        opacity=0.88,
        name='Frecuencia NPV',
    )
)

fig_npv.add_vline(
    x=mean_npv,
    line_dash='solid',
    line_color='#ff5252',
    line_width=2.5,
    annotation_text=f'Media: ${mean_npv:,.1f}M',
    annotation_position='top right',
    annotation_font_color='#ff5252',
)

fig_npv.add_vline(
    x=p10_npv,
    line_dash='dash',
    line_color='#ffb74d',
    line_width=2,
    annotation_text=f'P10: ${p10_npv:,.1f}M',
    annotation_position='top left',
    annotation_font_color='#ffb74d',
)

fig_npv.add_vline(
    x=p90_npv,
    line_dash='dash',
    line_color='#69f0ae',
    line_width=2,
    annotation_text=f'P90: ${p90_npv:,.1f}M',
    annotation_position='top right',
    annotation_font_color='#69f0ae',
)

fig_npv.update_layout(
    title=f'Distribución del Valor Presente Neto (NPV) — {N_SIMS:,} Simulaciones (WACC = {WACC:.0%})',
    xaxis_title='NPV (Millones de USD)',
    yaxis_title='Frecuencia (N° de escenarios)',
    template='plotly_dark',
    width=950,
    height=500,
    bargap=0.05,
    showlegend=False,
)
fig_npv.show()